In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('train.csv')

In [3]:
df.shape

(404351, 6)

In [4]:
df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [5]:
new_df = df.sample(30000)

In [6]:
new_df.isnull().sum()

id              0
qid1            0
qid2            0
question1       0
question2       0
is_duplicate    0
dtype: int64

In [7]:
new_df.duplicated().sum()

0

In [8]:
ques_df = new_df[['question1','question2']]
ques_df.head()

,question1,question2
298223,What is the relationship between monetary poli...,What is the difference between fiscal and mone...
239958,What're some good songs to make a lyric text p...,What's a good song to do a song lyrics prank?
243176,How much would it cost to start a TV channel?,How much money do I need to start a TV channel?
239483,Why was Quora named Quora?,"What does ""Quora"" mean?"
72985,How should I prepare for Accenture robotic aut...,How can I prepare for manual testing interview...


In [19]:
from sklearn.feature_extraction.text import CountVectorizer
# merge texts
questions = list(ques_df['question1']) + list(ques_df['question2'])

cv = CountVectorizer(max_features=300)
q1_arr, q2_arr = np.vsplit(cv.fit_transform(questions).toarray(),2)

In [20]:
temp_df1 = pd.DataFrame(q1_arr, index= ques_df.index)
temp_df2 = pd.DataFrame(q2_arr, index= ques_df.index)
temp_df = pd.concat([temp_df1, temp_df2], axis=1)
temp_df.shape

(30000, 600)

In [21]:
temp_df

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
298223,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
239958,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
243176,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
239483,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
72985,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
302584,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
249228,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
23866,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
223090,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [22]:
temp_df['is_duplicate'] = new_df['is_duplicate']

In [23]:
temp_df.head()

,0,1,2,3,4,5,6,7,8,9,...,291,292,293,294,295,296,297,298,299,is_duplicate
298223,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
239958,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
243176,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
239483,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
72985,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [24]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(temp_df.iloc[:,0:-1].values,temp_df.iloc[:,-1].values,test_size=0.2,random_state=42)

In [25]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test,y_pred)

0.7336666666666667

In [26]:
from xgboost import XGBClassifier
xgb = XGBClassifier()
xgb.fit(X_train,y_train)
y_pred = xgb.predict(X_test)
accuracy_score(y_test,y_pred)

0.7295